In [4]:
!pip install datasets
!pip uninstall -y protobuf grpcio grpcio-status


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 30.1 MB/s eta 0:00:00
Found existing installation: protobuf 6.31.1
Uninstalling protobuf-6.31.1:
  Successfully uninstalled protobuf-6.31.1
Found existing installation: grpcio 1.69.0
Uninstalling grpcio-1.69.0:
  Successfully uninstalled grpcio-1.69.0
Found existing installation: grpcio-status 1.62.3
Uninstalling grpcio-status-1.62.3:
  Successfully uninstalled grpcio-status-1.62.3


In [1]:
!pip install protobuf==3.20.3 grpcio==1.53.1 grpcio-status==1.48.2

In [1]:
from google.cloud import storage
import os

# Setup
#GCS_BUCKET = 'cdow'
#GCS_FOLDER = 'leads_env'
#LOCAL_BASE = f"/content/{GCS_FOLDER}"
#HF_CACHE = f"{LOCAL_BASE}/hf_cache"
#WHEELS_PATH = f"{LOCAL_BASE}/wheels"
#REQ_PATH = f"{LOCAL_BASE}/requirements.txt"

GCS_BUCKET = 'cdow'
GCS_FOLDER = 'leads_env'
LOCAL_BASE = f"/content/{GCS_FOLDER}"
HF_CACHE = f"{LOCAL_BASE}/hf_cache"
WHEELS_PATH = f"{LOCAL_BASE}/wheels"
REQ_PATH = f"{LOCAL_BASE}/requirements.txt"

In [4]:
#from google.cloud import storage
#import os

# Setup
GCS_BUCKET = 'cdow'
GCS_FOLDER = 'leads_env'
LOCAL_BASE = f"/content/{GCS_FOLDER}"
HF_CACHE = f"{LOCAL_BASE}/hf_cache"
WHEELS_PATH = f"{LOCAL_BASE}/wheels"
REQ_PATH = f"{LOCAL_BASE}/requirements.txt"


os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(WHEELS_PATH, exist_ok=True)

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

# Download requirements.txt
blob = bucket.blob(f'{GCS_FOLDER}/requirements.txt')
blob.download_to_filename(REQ_PATH)

print(' Found and downloaded requirements.txt')



# Download wheels
for blob in bucket.list_blobs(prefix=f'{GCS_FOLDER}/wheels'):
    relative_path = blob.name.replace(f'{GCS_FOLDER}/', '')  # Strip GCS_FOLDER prefix
    local_path = os.path.join(LOCAL_BASE, relative_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    blob.download_to_filename(local_path)
print('Wheels downloaded')

# Download Hugging Face model files
for blob in bucket.list_blobs(prefix=f'{GCS_FOLDER}/hf_cache'):
    relative_path = blob.name.replace(f'{GCS_FOLDER}/', '')
    local_path = os.path.join(LOCAL_BASE, relative_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    blob.download_to_filename(local_path)
print(' Hugging Face model files downloaded')


 Found and downloaded requirements.txt
Wheels downloaded
 Hugging Face model files downloaded


In [5]:
!pip install --no-index --find-links={WHEELS_PATH} -r {REQ_PATH}


Looking in links: /content/leads_env/wheels


In [2]:

#from google.cloud import bigquery
import os
import re

import sys
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from datasets import Dataset
from tab_transformer_pytorch import TabTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from collections import defaultdict
import pickle

from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import Dataset



import gcsfs
from itertools import islice
import gc
from datasets import load_dataset
# Initialize BigQuery client
# Set up GCS client
client = storage.Client()
bucket = client.bucket('cdow')
#print(TabTransformer)

In [3]:
fs = gcsfs.GCSFileSystem()

# Path to your CSV in GCS
parquet_path = 'cdow/leads_data/leads_scoring.parquet'
df = []
# Read CSV directly into a pandas DataFrame
with fs.open(f'gs://{parquet_path}') as f:
    df = pd.read_parquet(f)

In [4]:
print(df.head())
print(df.columns[0])
print(df.columns[3])
print(df.columns[4])
print(df.columns[5])
print(df.columns[8])
print(df.columns[15])
print(df.columns)
df_categories = df[[df.columns[0],df.columns[3],df.columns[4],df.columns[5],df.columns[8],df.columns[13],df.columns[15]]]
print(df_categories.head())
print(df_categories.dtypes)

               opp_id                          Email url     business_unit  \
0  006Hn00001PrjEeIAJ          iris.cruz@fda.hhs.gov  []  [MilliporeSigma]   
1  0061E00001LNU5MQAX        s.hemalatha@icar.gov.in  []            [APAC]   
2  0061E00001LlyCeQAJ  sraghoenath@immunoprecise.com  []  [MilliporeSigma]   
3  0061E00001MHptLQAT                rbtracy@uvm.edu  []  [MilliporeSigma]   
4  0061E00001LMN89QAH                wangbj5@163.com  []                []   

  Country EmailConsent search_terms linknames form_id business_field  \
0    [US]       [True]     [746436]        []      []             []   
1    [IN]       [True]           []        []      []             []   
2    [CA]      [False]           []        []      []             []   
3    [US]       [True]           []        []      []             []   
4      []      [False]           []        []      []             []   

  source_campaigns departments  \
0    [selfservice]          []   
1               []          []

In [5]:

max_list_lengths = {}
print(df_categories.columns)
for col in df_categories.columns[1:-1]:
    if df_categories[col].apply(lambda x: isinstance(x, list) or isinstance(x, object)).any():
        max_length = df_categories[col].apply(lambda x: len(x) if isinstance(x, list) or isinstance(x, object)  else 0).max()
        max_list_lengths[col] = max_length
for col, length in max_list_lengths.items():
    print(f"{col}: {length}")


#print(df.iloc[130:140])
print(len(df_categories))

Index(['opp_id', 'business_unit', 'Country', 'EmailConsent', 'form_id',
       'lead_source_most_recents', 'is_won'],
      dtype='object')
business_unit: 1
Country: 1
EmailConsent: 1
form_id: 26
lead_source_most_recents: 4
158464


In [6]:
print(df_categories.head())
print(print(df_categories.dtypes))

               opp_id     business_unit Country EmailConsent form_id  \
0  006Hn00001PrjEeIAJ  [MilliporeSigma]    [US]       [True]      []   
1  0061E00001LNU5MQAX            [APAC]    [IN]       [True]      []   
2  0061E00001LlyCeQAJ  [MilliporeSigma]    [CA]      [False]      []   
3  0061E00001MHptLQAT  [MilliporeSigma]    [US]       [True]      []   
4  0061E00001LMN89QAH                []      []      [False]      []   

  lead_source_most_recents  is_won  
0                  [Redox]   False  
1    [Other Digital Tools]   False  
2      [Technical Service]   False  
3   [Tradeshow/Conference]   False  
4                       []   False  
opp_id                      object
business_unit               object
Country                     object
EmailConsent                object
form_id                     object
lead_source_most_recents    object
is_won                        bool
dtype: object
None


In [7]:

for col in df_categories.columns[1:-1]:  # Assuming last column is boolean
    if pd.api.types.is_object_dtype(df_categories[col]):
        # Convert lists of strings to space-separated strings
        df_categories.loc[:, col] = df_categories[col].apply(
            lambda x: ' '.join(x) if isinstance(x, list) or isinstance(x, object) else x
        )
        # Fill NaNs with empty strings
        df_categories.loc[:, col]  = df_categories[col].fillna('')
        # Replace empty strings with "__MISSING__"
        df_categories.loc[:, col] = df_categories[col].replace('', '__MISSING__')


In [8]:

def count_special_values(df):
    def is_nan(val):
        try:
            return pd.isna(val)
        except:
            return False

    def is_empty_string(val):
        return isinstance(val, (str,object)) and val == ''

    def is_zero(val):
        return isinstance(val, (int, float,object)) and val == 0

    nan_counts = df.applymap(is_nan).sum(numeric_only=False)
    empty_string_counts = df.applymap(is_empty_string).sum(numeric_only=False)
    zero_counts = df.applymap(is_zero).sum(numeric_only=False)

    return pd.DataFrame({
        'NaNs': nan_counts,
        'Empty Strings': empty_string_counts,
        'Zeros': zero_counts
    })

summary = count_special_values(df_categories[df_categories.columns])
print(summary)



<ipython-input-8-3154274cdf64>:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nan_counts = df.applymap(is_nan).sum(numeric_only=False)
<ipython-input-8-3154274cdf64>:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  empty_string_counts = df.applymap(is_empty_string).sum(numeric_only=False)
<ipython-input-8-3154274cdf64>:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  zero_counts = df.applymap(is_zero).sum(numeric_only=False)


                          NaNs  Empty Strings  Zeros
opp_id                       0              0      0
business_unit                0              0      0
Country                      0              0      0
EmailConsent                 0              0      0
form_id                      0              0      0
lead_source_most_recents     0              0      0
is_won                       0              0  92318


In [9]:
# Initialize GCS filesystem
fs = gcsfs.GCSFileSystem()

# Path to your CSV in GCS
parquet_path = 'cdow/leads_data/leads_embeddings.parquet'
df_embeddings = []
# Read CSV directly into a pandas DataFrame
with fs.open(f'gs://{parquet_path}') as f:
    df_embeddings = pd.read_parquet(f)
print(len(df_embeddings))

158464


In [10]:
df_embeddings.drop('sample_key',inplace=True,axis=1)
print(df_embeddings.head())


              lead_id                                          embedding
0  006Hn00001PrjEeIAJ  [0.017692046239972115, 0.046732738614082336, 0...
1  0061E00001LNU5MQAX  [-0.015978705137968063, 0.03206974267959595, 0...
2  0061E00001LlyCeQAJ  [-0.011142889969050884, 0.03722440451383591, 0...
3  0061E00001MHptLQAT  [-0.0703454539179802, 0.011429915204644203, 0....
4  0061E00001LMN89QAH  [-0.008932134136557579, 0.04679758474230766, 0...


In [11]:
categorical_cols = df_categories.columns[1:-1]


set_df_embeddings = set(df_embeddings['lead_id'].to_numpy())
set_df_categories = set(df_categories['opp_id'].to_numpy())

# Find strings not common to both
unique_strings = list(set_df_embeddings.symmetric_difference(set_df_categories))

print(df_categories['opp_id'].duplicated().sum())  # Duplicates in df_categories
print(df_embeddings['lead_id'].duplicated().sum())  # Duplicates in df_embeddings

print(unique_strings)


501
501
[]


In [12]:

duplicates_categories = df_categories[df_categories.duplicated(subset='opp_id', keep=False)]
#print(duplicates_categories['opp_id'].value_counts())


duplicates_embeddings = df_embeddings[df_embeddings.duplicated(subset='lead_id', keep=False)]
#print(duplicates_embeddings['lead_id'].value_counts())
#HACK QUERIES <UST BE LOOKED AT

df_categories = df_categories.drop_duplicates(subset='opp_id', keep='first')
df_embeddings = df_embeddings.drop_duplicates(subset='lead_id', keep='first')

#print(df_categories[df_categories['opp_id'] == '006Hn00001Qe4HLIAZ'])
#print(df_embeddings[df_embeddings['lead_id'] == '006Hn00001Qe4HLIAZ'])
#print(duplicates_embeddings.loc['006Hn00001Lmp4KIAR',

In [13]:
duplicates_categories = df_categories[df_categories.duplicated(subset='opp_id', keep=False)]
print(duplicates_categories['opp_id'].value_counts())


duplicates_embeddings = df_embeddings[df_embeddings.duplicated(subset='lead_id', keep=False)]
print(duplicates_embeddings['lead_id'].value_counts())
print(len(df_categories))
print(len(df_embeddings))
#

Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)
157963
157963


In [14]:
#encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
#encoder.fit(df_categories[["country_identifier", "business_unit_identifier", "lead_source_identifier"]])


#categorical_cols = ["country_identifier", "business_unit_identifier", "lead_source_identifier"]

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(df_categories[categorical_cols])


categories = [len(cats) for cats in encoder.categories_]

#categories = [(len(cats), 32) for cats in encoder.categories_]


X_categorical = encoder.transform(df_categories[categorical_cols])
X_categorical_df = pd.DataFrame(X_categorical, columns=categorical_cols)
X_categorical_df['opp_id'] = df_categories['opp_id'].values
X_categorical_df['is_won'] = df_categories['is_won'].values

print(categories)
print(X_categorical_df.head())

[10, 99, 3, 1543, 295]
   business_unit  Country  EmailConsent  form_id  lead_source_most_recents  \
0            7.0     91.0           1.0   1542.0                     140.0   
1            0.0     43.0           1.0   1542.0                     109.0   
2            7.0     14.0           0.0   1542.0                     204.0   
3            7.0     91.0           1.0   1542.0                     226.0   
4            9.0     98.0           0.0   1542.0                     294.0   

               opp_id  is_won  
0  006Hn00001PrjEeIAJ   False  
1  0061E00001LNU5MQAX   False  
2  0061E00001LlyCeQAJ   False  
3  0061E00001MHptLQAT   False  
4  0061E00001LMN89QAH   False  


In [15]:
df_merged_training_data = pd.merge(X_categorical_df, df_embeddings, left_on='opp_id', right_on='lead_id', how='inner')
df_merged_training_data.drop('lead_id',inplace=True,axis=1)

print(df_merged_training_data.head())
print(len(df_merged_training_data))
print(categorical_cols)
print(df_merged_training_data.columns)

   business_unit  Country  EmailConsent  form_id  lead_source_most_recents  \
0            7.0     91.0           1.0   1542.0                     140.0   
1            0.0     43.0           1.0   1542.0                     109.0   
2            7.0     14.0           0.0   1542.0                     204.0   
3            7.0     91.0           1.0   1542.0                     226.0   
4            9.0     98.0           0.0   1542.0                     294.0   

               opp_id  is_won  \
0  006Hn00001PrjEeIAJ   False   
1  0061E00001LNU5MQAX   False   
2  0061E00001LlyCeQAJ   False   
3  0061E00001MHptLQAT   False   
4  0061E00001LMN89QAH   False   

                                           embedding  
0  [0.017692046239972115, 0.046732738614082336, 0...  
1  [-0.015978705137968063, 0.03206974267959595, 0...  
2  [-0.011142889969050884, 0.03722440451383591, 0...  
3  [-0.0703454539179802, 0.011429915204644203, 0....  
4  [-0.008932134136557579, 0.04679758474230766, 0...  
15

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
class HybridDataset(Dataset):
    def __init__(self, df,columns,device='cpu'):

        self.categoricals = df[columns].values
        self.embeddings = np.stack(df['embedding'].values)
        self.targets = df['is_won'].astype(int).values

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.categoricals[idx], dtype=torch.long).to(device),
            torch.tensor(self.embeddings[idx], dtype=torch.float).to(device),
            torch.tensor(self.targets[idx], dtype=torch.float).to(device)
        )


class ignoreTabTransformerEmbedder(TabTransformer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # fetch the combined embedding table
        if not hasattr(self, "category_embed"):
            raise AttributeError("No 'category_embed' found in TabTransformerEmbedder")
        if not isinstance(self.category_embed, nn.Embedding):
            raise TypeError("category_embed must be an nn.Embedding")

        self._cat_embed = self.category_embed

        # If you need per-column offsets, compute them here
        # using self.categories (list of cardinalities)
        import numpy as np
        self._offsets = torch.tensor(
            np.concatenate([[0], np.cumsum(self.categories)[:-1]]),
            dtype=torch.long
        )

    def forward(self, x_cat, x_cont=None):
        # add per-feature offsets
        x_cat = x_cat + self._offsets.to(x_cat.device)

        # single lookup into big embedding table
        cat_embeds = self._cat_embed(x_cat)
        # shape: (batch_size, num_categorical, embed_dim)

        # optional continuous features
        if self.num_continuous and x_cont is not None:
            cont = self.to_continuous(x_cont).unsqueeze(1)
            x    = torch.cat([cat_embeds, cont], dim=1)
        else:
            x = cat_embeds

        # pass through transformer & flatten
        x = self.transformer(x)
        return x.view(x.size(0), -1)



class TabTransformerEmbedder(TabTransformer):
    def __init__(
        self,
        categories,
        num_continuous=0,
        dim=32,
        depth=6,
        heads=8,
        dim_head=16,
        attn_dropout=0.0,
        ff_dropout=0.0,
        #mlp_dropout=0.0
    ):
        self.num_features = 32
        self.dim = 32
        # We call the parent to build embeddings + transformer blocks + MLP head
        # but we’ll override forward to bypass its MLP.
        super().__init__(
            categories=categories,
            num_continuous=num_continuous,
            dim=dim,
            depth=depth,
            heads=heads,
            dim_head=dim_head,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            mlp_hidden_mults=[1],    # dummy, we won’t use it
            #mlp_act='relu',
            #mlp_dropout=mlp_dropout,
            #num_classes=1            # dummy, output unused
        )

        if not hasattr(self, "category_embed"):
            raise AttributeError("No 'category_embed' found in TabTransformerEmbedder")
        if not isinstance(self.category_embed, nn.Embedding):
            raise TypeError("category_embed must be an nn.Embedding")
        self._cat_embeds = self.category_embed

        self.categories = categories
        self._cat_embeds = nn.ModuleList([
            nn.Embedding(cat, dim) for cat in categories
        ])

        self._offsets = torch.tensor(
            np.concatenate([[0], np.cumsum(self.categories)[:-1]]),
            dtype=torch.long
        )




    def forward(self, x_cat, x_cont=None):

        cat_embeds = torch.stack([
            emb(x_cat[:, i])
            for i, emb in enumerate(self._cat_embeds)
        ], dim=1)

        # 2) Optionally project continuous features, then concat
        if self.num_continuous and x_cont is not None:
            cont_embeds = self.to_continuous(x_cont)  # → (batch, dim)
            cont_embeds = cont_embeds.unsqueeze(1)    # → (batch, 1, dim)
            x = torch.cat([cat_embeds, cont_embeds], dim=1)
        else:
            x = cat_embeds                            # → (batch, num_cat, dim)

        # 3) Transformer blocks
        x = self.transformer(x)                      # → (batch, num_features, dim)
        # 4) Flatten to a feature vector
        batch_size = x.size(0)
        features = x.view(batch_size, -1)            # → (batch, num_features * dim)

        return features


class CombinedModel(nn.Module):
    def __init__(
        self,
        categories,
        num_continuous,
        other_dim=64,
        other_hidden_dim=128,
        text_embeddings_dim=768,
        final_output_dim=1
    ):
        super().__init__()

        # 2.1) The tabular embedder
        self.tab_emb = TabTransformerEmbedder(
            categories=categories,
            num_continuous=num_continuous,
            dim=32,
            depth=4,
            heads=8,
            dim_head=16,
            attn_dropout=0.1,
            ff_dropout=0.1,
            #mlp_dropout=0.1
        )
        self._cat_embed = self.tab_emb.category_embed
        self.text_embeddings_dim = text_embeddings_dim
        # Compute its output dimension

        #tab_out_dim = self.tab_emb.num_features * self.tab_emb.dim
        tab_out_dim = self.tab_emb.dim*len(categories)
        combined_in_dim = tab_out_dim + other_hidden_dim

        #for name, param in self.tab_emb.named_parameters():
        #    if param.requires_grad:
        #        print(name, param.shape)


        # 2.2) Project the “other” modality (e.g. pre-computed image features)
        self.other_proj = nn.Sequential(
            nn.Linear(text_embeddings_dim, other_hidden_dim),
            #nn.Linear(other_dim, other_hidden_dim),
            nn.ReLU()
        )

        # 2.3) Final head that fuses both representations
        self.combined_head = nn.Sequential(
            nn.Linear(tab_out_dim + other_hidden_dim, other_hidden_dim),
            #nn.Linear(tab_out_dim + other_hidden_dim, other_hidden_dim),
            nn.ReLU(),
            nn.Linear(other_hidden_dim, final_output_dim)
        )

        self.tab_norm = nn.LayerNorm(tab_out_dim)
        self.other_norm = nn.LayerNorm(other_hidden_dim)

    def forward(self, x_cat, x_cont,x_other):#, x_other):
        # TabTransformer features: (batch_size, num_features * dim)
        batch_size = x_cat.size(0)

        # Dummy continuous input (none in your case)
        x_cont = torch.zeros((batch_size, 0), device=x_cat.device)
        tab_features = self.tab_emb(x_cat, x_cont)
        other_features = self.other_proj(x_other)

        tab_features = self.tab_norm(tab_features)
        other_features = self.other_norm(other_features)

        merged = torch.cat([tab_features, other_features], dim=1)
        out    = self.combined_head(merged)

        return out













cuda


In [ ]:
dataset = HybridDataset(df_merged_training_data, columns=categorical_cols,device=device)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
categories = [len(cats) for cats in encoder.categories_]

model = CombinedModel(categories=categories,num_continuous=0).to(device)

model.train()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for epoch in range(50):  # adjust epochs
    for x_cat, x_embed, y in dataloader:
        x_cat = x_cat.to(device)
        x_embed = x_embed.to(device)
        y = y.to(device).unsqueeze(1)
        batch_size = x_cat.size(0)

        # Dummy continuous input (none in your case)
        x_cont = torch.zeros((batch_size, 0), device=x_cat.device)
        optimizer.zero_grad()
        preds = model(x_cat,x_cont, x_embed)
        probs = torch.sigmoid(preds)

        loss = criterion(probs, y)
        loss.backward()

        #for name, param in model.named_parameters():
        #  if param.requires_grad:
        #      if param.grad is not None:
        #          print(f"{name} has gradient with mean {param.grad.mean().item():.6f}")
        #      else:
        #          print(f"{name} has NO gradient")
        #break

        optimizer.step()

    #break
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")




Epoch 1, Loss: 0.5481
Epoch 2, Loss: 0.6277
Epoch 3, Loss: 0.6231
Epoch 4, Loss: 0.7182


In [ ]:
#class HybridTabTransformerModel(nn.Module):
#    def __init__(self, categories, embedding_dim, transformer_dim=32, num_classes=1):
#        super().__init__()

#        self.tab_transformer = TabTransformerEmbedder(
#            num_continuous=0,
#            categories=categories,  # List of (num_categories, embedding_dim)
            #dim=transformer_dim,
#            depth=6,
#            heads=8,
#            attn_dropout=0.1,
#            ff_dropout=0.1
#        )


        # Final classifier after combining TabTransformer output and external embeddings
#        self.embedding_dim = embedding_dim
#        self.transformer_dim = transformer_dim

#        self.classifier = nn.Sequential(
#            nn.Linear(transformer_dim + embedding_dim, 64),
#            nn.ReLU(),
#            nn.Dropout(0.2),
#            nn.Linear(64, num_classes),
#            nn.Sigmoid()  # For binary classification
#        )

#    def forward(self, x_cat, x_embed):
#        batch_size = x_cat.size(0)

        # Dummy continuous input (none in your case)
#        x_cont = torch.zeros((batch_size, 0), device=x_cat.device)

        # Get transformer output
#        tab_out = self.tab_transformer(x_cat, x_cont)

        # If tab_out is [batch_size, 1], it's likely pooled or reduced — fix it
#        if tab_out.ndim == 3:
#            tab_out = tab_out.mean(dim=1)  # mean pooling across tokens
#        elif tab_out.shape[1] == 1:
#            tab_out = tab_out.squeeze(1)  # remove singleton dimension

        # Concatenate with external embeddings
#        combined = torch.cat([tab_out, x_embed], dim=1)

        # Final prediction
#        return self.classifier(combined)

11
100
296


In [ ]:
import json
torch.save(model.state_dict(), "./tab_transformer_lead_scoring.pt")

model_config = {
    "categories": categories,
    "num_continuous": 0,
    "dim": 32,
    "depth": 6,
    "heads": 8,
    "attn_dropout": 0.1,
    "ff_dropout": 0.1,
    "mlp_hidden_mults": [4, 2],
    "mlp_act": "ReLU"
}

with open("tab_transformer_config.json", "w") as f:
    json.dump(model_config,f)


with open("./le_country.pkl", "wb") as f:
    pickle.dump(le_country, f)

with open("./le_bu.pkl", "wb") as f:
    pickle.dump(le_bu, f)

with open("./le_ls.pkl", "wb") as f:
    pickle.dump(le_ls, f)


client = storage.Client()

# Define bucket and blob
bucket_name = "cdow"
destination_blob_name = "leads_data/le_country.pkl"
local_file_path = "./le_country.pkl"
bucket = client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(local_file_path)

destination_blob_name = "leads_data/le_bu.pkl"
local_file_path = "./le_bu.pkl"

bucket = client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(local_file_path)

destination_blob_name = "leads_data/le_ls.pkl"
local_file_path = "./le_ls.pkl"
bucket = client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(local_file_path)



blob_weights = bucket.blob("leads_data/tab_transformer_weights.pth")
blob_weights.upload_from_filename("tab_transformer_weights.pth")

# Upload config
blob_config = bucket.blob("leads_data/tab_transformer_config.json")
blob_config.upload_from_filename("tab_transformer_config.json")

#for epoch in range(10):
#    model.train()
#    prob = nn.sigmoid(model(X_cat, X_cont).squeeze())
#    loss = loss_fn(logits, y.float())
#    loss.backward()
#    optimizer.step()
#    optimizer.zero_grad()
#    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# === Save Model and Metadata ===
#torch.save(model.state_dict(), "tabtransformer.pt")
#with open("./tabtransformer_meta.pkl", "wb") as f:
#    pickle.dump({
#        "category_dims": category_dims,
#        "num_continuous": num_continuous
#    }, f)


In [ ]:

model = TabTransformer(
    categories=categories,
    num_continuous=0,
    dim=32,
    depth=6,
    heads=8,
    attn_dropout=0.1,
    ff_dropout=0.1,
    mlp_hidden_mults=(4, 2),
    mlp_act=nn.ReLU()
)
model.load_state_dict(torch.load("tab_transformer_weights.pth"))
model = model.to(device)
model.eval()


In [ ]:


encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(df_categories[["country_identifier", "business_unit_identifier", "lead_source_identifier"]])


categorical_cols = ["country_identifier", "business_unit_identifier", "lead_source_identifier"]

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(df_categories[categorical_cols])


categories = [len(cats) for cats in encoder.categories_]

#import pickle


#with open("ordinal_encoder.pkl", "rb") as f:
#    encoder = pickle.load(f)

# Then use it to transform new data
#new_data[categorical_cols] = encoder.transform(new_data[categorical_cols])

